<a href="https://colab.research.google.com/github/Tdhairya2325/UAV-Intelligence-System/blob/main/notebooks/02_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **02 — Data Preprocessing**
## **UAV Intelligence System**

## Dataset: VisDrone2019-DET

Objective: Clean, transform and prepare aerial image annotations for YOLO object detection training.

### **Section	Topic**
1	Environment setup

2	Load dataset paths

3	Understand YOLO annotation format

4	Convert VisDrone bounding boxes

5	Handle ignored regions

6	Create YOLO folder structure

7	Generate label files

8	Validate converted annotations

9	Visualize converted YOLO labels

10	Prepare dataset configuration YAML

# **Environment Setup**

In [1]:
# ==========================================
# UAV INTELLIGENCE SYSTEM
# NOTEBOOK 02 - DATA PREPROCESSING
# ==========================================

# 1. Import libraries
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. Define project directories
PROJECT_DIR = "/content/drive/MyDrive/UAV-Intelligence-System"

RAW_DIR = os.path.join(PROJECT_DIR, "data", "raw")

PROCESSED_DIR = os.path.join(PROJECT_DIR, "data", "processed")

# 4. Create processed directory if it doesn't exist
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Environment initialized successfully!")
print("Project directory:", PROJECT_DIR)

Mounted at /content/drive
Environment initialized successfully!
Project directory: /content/drive/MyDrive/UAV-Intelligence-System


# **Load the dataset**

In [2]:
# ==========================================
# LOAD DATASET PATHS
# ==========================================

# Locate training and validation folders
train_matches = glob.glob(
    os.path.join(RAW_DIR, "VisDrone2019-DET-train*")
)

val_matches = glob.glob(
    os.path.join(RAW_DIR, "VisDrone2019-DET-val*")
)

# Check whether folders exist
if not train_matches or not val_matches:
    raise FileNotFoundError("VisDrone dataset folders not found!")

TRAIN_DIR = train_matches[0]
VAL_DIR = val_matches[0]

# Load training images
train_images = sorted(
    glob.glob(os.path.join(TRAIN_DIR, "images", "*.jpg"))
)

# Load validation images
val_images = sorted(
    glob.glob(os.path.join(VAL_DIR, "images", "*.jpg"))
)

# Load annotation file paths
train_annotations = sorted(
    glob.glob(os.path.join(TRAIN_DIR, "annotations", "*.txt"))
)

val_annotations = sorted(
    glob.glob(os.path.join(VAL_DIR, "annotations", "*.txt"))
)

# Load cleaned training annotations CSV
cleaned_path = os.path.join(
    PROCESSED_DIR,
    "train_annotations_cleaned.csv"
)

annotations_cleaned = pd.read_csv(cleaned_path)

# Display dataset information
print("========== DATASET INFORMATION ==========")

print("Training images:", len(train_images))
print("Training annotation files:", len(train_annotations))

print("Validation images:", len(val_images))
print("Validation annotation files:", len(val_annotations))

print("Cleaned training annotation rows:", len(annotations_cleaned))

print("==========================================")

========== DATASET INFORMATION ==========
Training images: 6471
Training annotation files: 6471
Validation images: 548
Validation annotation files: 548
Cleaned training annotation rows: 353547


# **Understand YOLO annotation format**

In [4]:
def convert_to_yolo(x, y, w, h, image_width, image_height, category):

    # 1. Check bounding-box dimensions
    if w <= 0 or h <= 0:
        return None

    # 2. Check image boundaries
    if (
        x < 0 or
        y < 0 or
        x + w > image_width or
        y + h > image_height
    ):
        return None

    # 3. Convert VisDrone category to YOLO class ID
    class_id = int(category) - 1

    # 4. Calculate center coordinates
    x_center = x + (w / 2)
    y_center = y + (h / 2)

    # 5. Normalize coordinates
    x_center_norm = x_center / image_width
    y_center_norm = y_center / image_height

    width_norm = w / image_width
    height_norm = h / image_height

    # 6. Return YOLO annotation
    return (
        f"{class_id} "
        f"{x_center_norm:.6f} "
        f"{y_center_norm:.6f} "
        f"{width_norm:.6f} "
        f"{height_norm:.6f}"
    )

In [5]:
# Sample VisDrone annotation
x = 440
y = 541
w = 271
h = 152
category = 6

# Image dimensions
image_width = 1400
image_height = 1050

# Convert annotation
yolo_label = convert_to_yolo(
    x, y, w, h,
    image_width,
    image_height,
    category
)

print("Converted YOLO annotation:")
print(yolo_label)

Converted YOLO annotation:
5 0.411071 0.587619 0.193571 0.144762


In [6]:
invalid_label = convert_to_yolo(
    x=545,
    y=414,
    w=10,
    h=0,
    image_width=1400,
    image_height=1050,
    category=1
)

print(invalid_label)

None


In [7]:
yolo_class_names = [
    "Pedestrian",
    "People",
    "Bicycle",
    "Car",
    "Van",
    "Truck",
    "Tricycle",
    "Awning-tricycle",
    "Bus",
    "Motor"
]

for class_id, class_name in enumerate(yolo_class_names):
    print(class_id, ":", class_name)

0 : Pedestrian
1 : People
2 : Bicycle
3 : Car
4 : Van
5 : Truck
6 : Tricycle
7 : Awning-tricycle
8 : Bus
9 : Motor


In [9]:
# Select one training image
sample_image_path = train_images[0]

# Read its dimensions
from PIL import Image

with Image.open(sample_image_path) as img:
    image_width, image_height = img.size

# Find corresponding annotation file
sample_annotation_path = os.path.join(
    TRAIN_DIR,
    "annotations",
    os.path.basename(sample_image_path).replace(".jpg", ".txt")
)

# Read annotation
sample_df = pd.read_csv(
    sample_annotation_path,
    header=None
)

sample_df = sample_df.dropna(axis=1, how="all")
sample_df.columns = [
    "left", "top", "width", "height",
    "score", "category", "truncation", "occlusion"
]

# Convert valid annotations
sample_yolo_labels = []

for _, row in sample_df.iterrows():

    # Ignore score-0 regions
    if row["score"] != 1:
        continue

    # Keep only the ten VisDrone object categories
    if row["category"] not in range(1, 11):
        continue

    label = convert_to_yolo(
        row["left"],
        row["top"],
        row["width"],
        row["height"],
        image_width,
        image_height,
        row["category"]
    )

    if label is not None:
        sample_yolo_labels.append(label)

print("Image:", os.path.basename(sample_image_path))
print("Image dimensions:", image_width, "x", image_height)
print("YOLO annotations generated:", len(sample_yolo_labels))

print("\nFirst five YOLO labels:")

for label in sample_yolo_labels[:5]:
    print(label)

Image: 0000002_00005_d_0000014.jpg
Image dimensions: 960 x 540
YOLO annotations generated: 82

First five YOLO labels:
3 0.776042 0.902778 0.077083 0.061111
3 0.697396 0.829630 0.063542 0.085185
3 0.652083 0.786111 0.066667 0.094444
3 0.617188 0.757407 0.063542 0.070370
3 0.596354 0.719444 0.067708 0.061111
